## Funnel Analysis

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# paths
MASTER = r'C:\Users\dell\Desktop\smart-home-product-analysis\datasets\master_tables'

# load master tables
session_master  = pd.read_csv(os.path.join(MASTER, 'session_master.csv'))
behavior_master = pd.read_csv(os.path.join(MASTER, 'behavior_master.csv'))
order_master    = pd.read_csv(os.path.join(MASTER, 'order_master.csv'))

print("session_master  :", session_master.shape)
print("behavior_master :", behavior_master.shape)
print("order_master    :", order_master.shape)

session_master  : (146000, 21)
behavior_master : (146000, 14)
order_master    : (22406, 23)


In [77]:
# basic kpis
total_sessions    = len(session_master)
total_users       = session_master['user_id'].nunique()
total_orders      = session_master['is_converted'].sum()
conversion_rate   = round(session_master['is_converted'].mean() * 100, 2)

# revenue from order master — one row per product so need order level revenue
order_level       = order_master.drop_duplicates(subset=['order_id'])
total_revenue     = round(order_level['net_revenue'].sum(), 2)
avg_order_value   = round(order_level['net_revenue'].mean(), 2)
revenue_per_session = round(total_revenue / total_sessions, 2)

print("total sessions      :", total_sessions)
print("total unique users  :", total_users)
print("total orders        :", total_orders)
print("conversion rate     :", conversion_rate, "%")
print("total revenue       :", "$", total_revenue)
print("avg order value     :", "$", avg_order_value)
print("revenue per session :", "$", revenue_per_session)

total sessions      : 146000
total unique users  : 102435
total orders        : 11206
conversion rate     : 7.68 %
total revenue       : $ 5440694.0
avg order value     : $ 485.52
revenue per session : $ 37.27


146,000 sessions but only 102,435 unique users 
1. some users visited multiple times before buying 
2. remarketing and retargeting is likely working. 

7.68% conversion rate 
1. above industry average of 2-4% 
2.  visitors have high purchase intent 

AOV of $485 is high 
1. smart home products are considered purchases 
2. people research before buying 

$37 revenue per session 
1. every visit generates value 
2. increasing traffic will directly increase revenue 

In [78]:
print(behavior_master.columns.tolist())

['session_id', 'user_id', 'total_pageviews', 'unique_pages_visited', 'visited_product_page', 'visited_cart_page', 'visited_checkout_page', 'visited_category_page', 'total_clicks', 'clicked_add_to_cart', 'clicked_buy_now', 'avg_scroll_percent', 'max_scroll_percent', 'scrolled_75_percent']


In [79]:
# merging session master and behavior master on session_id
df = session_master.merge(behavior_master, on='session_id', how='left')

# funnel steps — each number is count of sessions that reached that step
funnel = {
    'Session Started'       : len(df),
    'Visited Category Page' : df['visited_category_page'].sum(),
    'Visited Product Page'  : df['visited_product_page'].sum(),
    'Clicked Add to Cart'   : df['clicked_add_to_cart'].sum(),
    'Added to Cart'         : df['visited_cart_page'].sum(),
    'Reached Checkout'      : df['visited_checkout_page'].sum(),
    'Placed Order'          : df['is_converted'].sum()
}

funnel_df = pd.DataFrame(list(funnel.items()), columns=['stage', 'sessions'])

# drop off from previous step
funnel_df['drop_off']      = funnel_df['sessions'].diff().fillna(0).abs().astype(int)
funnel_df['drop_off_pct']  = (funnel_df['drop_off'] / funnel_df['sessions'].shift(1) * 100).fillna(0).round(2)
funnel_df['overall_pct']   = (funnel_df['sessions'] / funnel_df['sessions'].iloc[0] * 100).round(2)

print(funnel_df)

                   stage  sessions  drop_off  drop_off_pct  overall_pct
0        Session Started    146000         0          0.00       100.00
1  Visited Category Page     38036    107964         73.95        26.05
2   Visited Product Page    146000    107964        283.85       100.00
3    Clicked Add to Cart     47376     98624         67.55        32.45
4          Added to Cart     37775      9601         20.27        25.87
5       Reached Checkout     59373     21598         57.18        40.67
6           Placed Order     11206     48167         81.13         7.68


In [80]:
# check landing pages to understand why product page = 100%
print(session_master['landing_page'].value_counts().head(10))

# check how many sessions skipped cart but reached checkout
skipped_cart = df[(df['visited_cart_page'] == 0) & (df['visited_checkout_page'] == 1)]
print(f"sessions that skipped cart but reached checkout: {len(skipped_cart)}")

landing_page
/products/ring-alarm-8-piece      36779
/products/video-doorbell-pro-2    36742
/products/stick-up-cam-battery    36262
/products/indoor-cam-(2nd-gen)    36217
Name: count, dtype: int64
sessions that skipped cart but reached checkout: 35789


1. All Traffic Lands Directly on Product Pages
every session lands on a product page — no homepage traffic at all. the brand is sending paid ads directly to specific product pages which means visitors arrive with high purchase intent.
2. Two Purchase Paths Exist
35,789 sessions skipped cart and went directly to checkout — confirming two paths exist, standard Add to Cart flow and a Buy Now flow. analysing them as one funnel was giving misleading numbers.
3. Checkout Abandonment is the Biggest Problem
81% of users who reached checkout did not complete the purchase — this is the single biggest drop off in the entire funnel and the highest priority problem.
4. Category Pages are Barely Used
only 26% of sessions visited a category page — most users go straight from product page to purchase without browsing other products.
5. Add to Cart Drop
20% of users who clicked add to cart never visited the cart page — they clicked but didn't follow through.

In [81]:
# merging session and behavior master
df = session_master.merge(behavior_master, on='session_id', how='left')

# path 1 — standard add to cart flow
path1 = {
    'Session Started'      : len(df),
    'Visited Product Page' : int(df['visited_product_page'].sum()),
    'Clicked Add to Cart'  : int(df['clicked_add_to_cart'].sum()),
    'Visited Cart'         : int(df['visited_cart_page'].sum()),
    'Reached Checkout'     : int(df['visited_checkout_page'].sum()),
    'Placed Order'         : int(df['is_converted'].sum())
}

# path 2 — buy now flow
path2 = {
    'Session Started'      : len(df),
    'Visited Product Page' : int(df['visited_product_page'].sum()),
    'Clicked Buy Now'      : int(df['clicked_buy_now'].sum()),
    'Reached Checkout'     : int(df['visited_checkout_page'].sum()),
    'Placed Order'         : int(df['is_converted'].sum())
}

# convert to dataframes
path1_df = pd.DataFrame(list(path1.items()), columns=['stage', 'sessions'])
path2_df = pd.DataFrame(list(path2.items()), columns=['stage', 'sessions'])

# drop off from previous step
path1_df['drop_off_pct'] = (path1_df['sessions'].diff().abs() / path1_df['sessions'].shift(1) * 100).fillna(0).round(2)
path2_df['drop_off_pct'] = (path2_df['sessions'].diff().abs() / path2_df['sessions'].shift(1) * 100).fillna(0).round(2)

# overall % from session started
path1_df['overall_pct'] = (path1_df['sessions'] / len(df) * 100).round(2)
path2_df['overall_pct'] = (path2_df['sessions'] / len(df) * 100).round(2)

print("PATH 1 — Add to Cart Flow")
print(path1_df.to_string(index=False))

print("\nPATH 2 — Buy Now Flow")
print(path2_df.to_string(index=False))

PATH 1 — Add to Cart Flow
               stage  sessions  drop_off_pct  overall_pct
     Session Started    146000          0.00       100.00
Visited Product Page    146000          0.00       100.00
 Clicked Add to Cart     47376         67.55        32.45
        Visited Cart     37775         20.27        25.87
    Reached Checkout     59373         57.18        40.67
        Placed Order     11206         81.13         7.68

PATH 2 — Buy Now Flow
               stage  sessions  drop_off_pct  overall_pct
     Session Started    146000          0.00       100.00
Visited Product Page    146000          0.00       100.00
     Clicked Buy Now     47565         67.42        32.58
    Reached Checkout     59373         24.82        40.67
        Placed Order     11206         81.13         7.68


What I Found
1. All traffic lands directly on product pages
every session starts on a product page — the brand drives paid traffic directly to specific products, not the homepage. visitors arrive already knowing what they want.
2. Two purchase paths exist
both paths have almost equal usage — 47,376 vs 47,565 clicks showing users are split evenly between both.
3. Buy Now is more efficient
buy now has only one drop off point between clicking and checkout vs two drop off points in the add to cart path. fewer steps means fewer chances to abandon.
4. Checkout abandonment is the real problem
both paths lose 81% of users at checkout — this is consistent across both paths meaning the problem is the checkout experience itself not the path taken to get there.
5. 67% of sessions never engage with any purchase button
out of 146,000 sessions only 47,000 clicked either button — meaning most visitors browse the product page and leave without taking any purchase action.

In [82]:
# split into converted and not converted
df = session_master.merge(behavior_master, on='session_id', how='left')

converted     = df[df['is_converted'] == 1]
not_converted = df[df['is_converted'] == 0]

# compare behavior between the two groups
metrics = {
    'avg pageviews'          : ('total_pageviews',        False),
    'avg unique pages'       : ('unique_pages_visited',   False),
    'avg clicks'             : ('total_clicks',           False),
    'avg scroll %'           : ('avg_scroll_percent',     False),
    'avg max scroll %'       : ('max_scroll_percent',     False),
    '% visited product page' : ('visited_product_page',   True),
    '% visited cart'         : ('visited_cart_page',      True),
    '% visited checkout'     : ('visited_checkout_page',  True),
    '% clicked add to cart'  : ('clicked_add_to_cart',    True),
    '% clicked buy now'      : ('clicked_buy_now',        True),
    '% scrolled 75%+'        : ('scrolled_75_percent',    True),
}

rows = []
for label, (col, is_pct) in metrics.items():
    c  = round(converted[col].mean() * (100 if is_pct else 1), 2)
    nc = round(not_converted[col].mean() * (100 if is_pct else 1), 2)
    rows.append([label, c, nc, round(c - nc, 2)])

comparison = pd.DataFrame(rows, columns=['metric', 'converted', 'not_converted', 'difference'])
print(comparison.to_string(index=False))

                metric  converted  not_converted  difference
         avg pageviews       3.49           2.34        1.15
      avg unique pages       2.81           1.98        0.83
            avg clicks       2.44           1.32        1.12
          avg scroll %      62.58          62.53        0.05
      avg max scroll %      85.73          85.60        0.13
% visited product page     100.00         100.00        0.00
        % visited cart      44.85          24.30       20.55
    % visited checkout      71.43          38.11       33.32
 % clicked add to cart      56.60          30.44       26.16
     % clicked buy now      57.17          30.53       26.64
       % scrolled 75%+      81.11          43.27       37.84


1. Buyers browse more — converted sessions viewed 3.49 pages and clicked 2.44 times vs 2.34 pages and 1.32 clicks for non converted. higher engagement = higher purchase intent.
2. Average scroll means nothing — 62.58% vs 62.53% — almost identical between both groups. scrolling alone does not predict purchase.
3. Deep scrolling does — 81% of buyers scrolled 75%+ vs only 43% of non buyers. users who read pages deeply are far more likely to purchase.
4. Clicking purchase buttons is the strongest signal — 57% of buyers clicked add to cart or buy now vs only 30% of non buyers. these clicks are the clearest purchase intent indicators in the data.
5. Reaching checkout separates buyers from browsers — 71% of converted sessions reached checkout vs 38% of non converted. getting a user to checkout is the most critical step in the funnel.

In [83]:
# channel performance analysis
channel_perf = session_master.groupby('channel').agg(
    total_sessions    = ('session_id',    'count'),
    converted         = ('is_converted',  'sum'),
    total_revenue     = ('net_revenue',   'sum')
).reset_index()

# conversion rate and avg order value per channel
channel_perf['conversion_rate'] = (channel_perf['converted'] / channel_perf['total_sessions'] * 100).round(2)
channel_perf['avg_order_value'] = (channel_perf['total_revenue'] / channel_perf['converted']).round(2)
channel_perf['revenue_per_session'] = (channel_perf['total_revenue'] / channel_perf['total_sessions']).round(2)

# sort by total sessions
channel_perf = channel_perf.sort_values('total_sessions', ascending=False).reset_index(drop=True)

print(channel_perf.to_string(index=False))

    channel  total_sessions  converted  total_revenue  conversion_rate  avg_order_value  revenue_per_session
Paid Social           55930       4249     2078091.64             7.60           489.08                37.16
      Email           22537       1737      830028.21             7.71           477.85                36.83
Paid Search           11397        865      420034.42             7.59           485.59                36.85
    Display           11299        877      420243.81             7.76           479.18                37.19
        SMS           11298        903      434593.03             7.99           481.28                38.47
  Affiliate           11211        876      428743.74             7.81           489.43                38.24
    Organic           11181        826      399495.36             7.39           483.65                35.73
   Internal           11147        873      429463.79             7.83           491.94                38.53


Assumption: no direct traffic exists in this dataset. all sessions have utm tracking applied meaning every visit came from a tracked marketing campaign. the referrer = 'direct' pattern observed is due to referrer stripping by browsers and apps, not actual direct visits. 

Key Findings
1. Channel mix is heavily skewed
Paid Social = 38% of all sessions
all other channels = ~8% each
over-reliance on one channel is risky
2. Conversion rates are flat across channels
difference between best (SMS 7.99%) 
and worst (Organic 7.39%) is only 0.6%
channel does not significantly affect conversion
3. Revenue per session is similar everywhere
ranges from $35.73 to $38.53
no channel dramatically outperforms others

What I Would Recommend:
Scale SMS-highest conversion ratemost efficient channel
Diversify from Paid Social-38% dependency is riskytoo reliant on one channel
Improve Organic-lowest performanceSEO opportunity
Leverage Internal-highest AOVexisting customers spend more

In [87]:
# purchase intent signals
df = session_master.merge(behavior_master, on='session_id', how='left')

# signal 1 — clicked add to cart
signal1 = df.groupby('clicked_add_to_cart')['is_converted'].mean().mul(100).round(2)
print("conversion rate by add to cart click:")
print(signal1)

# signal 2 — clicked buy now
signal2 = df.groupby('clicked_buy_now')['is_converted'].mean().mul(100).round(2)
print("\nconversion rate by buy now click:")
print(signal2)

# signal 3 — scrolled 75%+
signal3 = df.groupby('scrolled_75_percent')['is_converted'].mean().mul(100).round(2)
print("\nconversion rate by scroll 75%+:")
print(signal3)

# signal 4 — visited cart page
signal4 = df.groupby('visited_cart_page')['is_converted'].mean().mul(100).round(2)
print("\nconversion rate by cart page visit:")
print(signal4)

# signal 5 — visited checkout page
signal5 = df.groupby('visited_checkout_page')['is_converted'].mean().mul(100).round(2)
print("\nconversion rate by checkout page visit:")
print(signal5)

conversion rate by add to cart click:
clicked_add_to_cart
0     4.93
1    13.39
Name: is_converted, dtype: float64

conversion rate by buy now click:
clicked_buy_now
0     4.88
1    13.47
Name: is_converted, dtype: float64

conversion rate by scroll 75%+:
scrolled_75_percent
0     2.69
1    13.48
Name: is_converted, dtype: float64

conversion rate by cart page visit:
visited_cart_page
0     5.71
1    13.31
Name: is_converted, dtype: float64

conversion rate by checkout page visit:
visited_checkout_page
0     3.70
1    13.48
Name: is_converted, dtype: float64


a user who scrolls deeply, clicks a purchase button and reaches checkout is highly likely to convert — these three signals together define a high intent user.

In [88]:
# revenue analysis
order_level = order_master.drop_duplicates(subset=['order_id'])

# revenue by channel
channel_revenue = order_level.groupby('channel').agg(
    total_orders    = ('order_id',     'count'),
    total_revenue   = ('net_revenue',  'sum'),
    avg_order_value = ('net_revenue',  'mean'),
    avg_discount    = ('discount',     'mean'),
    coupon_orders   = ('has_coupon',   'sum')
).reset_index()

channel_revenue['coupon_rate']        = (channel_revenue['coupon_orders'] / channel_revenue['total_orders'] * 100).round(2)
channel_revenue['avg_order_value']    = channel_revenue['avg_order_value'].round(2)
channel_revenue['avg_discount']       = channel_revenue['avg_discount'].round(2)
channel_revenue['total_revenue']      = channel_revenue['total_revenue'].round(2)
channel_revenue = channel_revenue.sort_values('total_revenue', ascending=False).reset_index(drop=True)

print("revenue by channel:")
print(channel_revenue.to_string(index=False))

# revenue by product
product_revenue = order_master.groupby('product_name').agg(
    total_orders  = ('order_id',      'count'),
    total_revenue = ('line_total',    'sum'),
    avg_price     = ('product_price', 'mean'),
    total_qty     = ('product_qty',   'sum')
).reset_index()

product_revenue['total_revenue'] = product_revenue['total_revenue'].round(2)
product_revenue['avg_price']     = product_revenue['avg_price'].round(2)
product_revenue = product_revenue.sort_values('total_revenue', ascending=False).reset_index(drop=True)

print("\nrevenue by product:")
print(product_revenue.to_string(index=False))

# coupon impact
coupon = order_level.groupby('has_coupon').agg(
    total_orders    = ('order_id',    'count'),
    total_revenue   = ('net_revenue', 'sum'),
    avg_order_value = ('net_revenue', 'mean'),
    avg_discount    = ('discount',    'mean')
).reset_index()

coupon['avg_order_value'] = coupon['avg_order_value'].round(2)
coupon['avg_discount']    = coupon['avg_discount'].round(2)
coupon['total_revenue']   = coupon['total_revenue'].round(2)

print("\ncoupon impact:")
print(coupon.to_string(index=False))

revenue by channel:
    channel  total_orders  total_revenue  avg_order_value  avg_discount  coupon_orders  coupon_rate
Paid Social          4249     2078091.64           489.08          9.95           2865        67.43
      Email          1737      830028.21           477.85         10.11           1177        67.76
        SMS           903      434593.03           481.28          9.84            613        67.88
   Internal           873      429463.79           491.94         10.42            583        66.78
  Affiliate           876      428743.74           489.43         10.21            588        67.12
    Display           877      420243.81           479.18         10.11            602        68.64
Paid Search           865      420034.42           485.59          9.77            582        67.28
     Direct           826      399495.36           483.65         10.28            545        65.98

revenue by product:
        product_name  total_orders  total_revenue  avg_pric

Revenue Analysis 

1. Paid Social drives the most revenue ($2.07M) — purely because of volume, not because buyers spend more. AOV is similar across all channels.
2. Internal channel has the highest AOV ($491.94) — returning customers who come through internal links spend the most per order.
3. Top 2 products drive 77% of total revenue — Video Doorbell Pro 2 ($2.11M) and Ring Alarm 8-piece ($2.09M) are the core revenue drivers. both priced at $249.99.
4. All 4 products have similar order volumes (~5,600 each) — revenue difference is purely driven by price, not demand.
5. 67% of all orders use a coupon — coupons are a core part of the purchase experience, consistent across every channel.
6. Coupons don't hurt AOV — only $3 difference between coupon ($484) and non coupon ($487) orders. coupons convert hesitant buyers without reducing spend.

Recommendations
1. Prioritize Video Doorbell and Ring Alarm in marketing — they drive most revenue and have equal demand.
2. Bundle Indoor Cam with premium products — same demand as expensive products but 4x less revenue, bundling could increase AOV.
3. Protect coupon strategy — removing coupons would hurt order volume without meaningfully improving revenue quality.